# 析锂无损检测工具链 · 快速上手

本 notebook 演示合成真值验证、真实数据检测与结果查看。零深度学习、零模型训练。

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from plating.config import load_config
from plating.synthetic import generate_dataset, ground_truth
from plating.methods import run_all, METHOD_NAMES

cfg = load_config()
print('方法：', METHOD_NAMES)

## 1. 合成数据真值验证

生成 8 个电芯（4 析锂、4 对照），逐方法统计灵敏度/特异度。

In [ ]:
cells = generate_dataset(cfg, n_cells=8, n_cycles=150)
gt = ground_truth(cells)
for m in METHOD_NAMES:
    tp = fn = fp = tn = 0
    for cell in cells:
        truth = gt[cell.cell_id]['plating']
        pred = any(r.positive for r in run_all(cell, cfg)[m])
        if truth:
            tp, fn = (tp+1, fn) if pred else (tp, fn+1)
        else:
            fp, tn = (fp+1, tn) if pred else (fp, tn+1)
    print(f'{m}: 灵敏度={tp/max(tp+fn,1):.2f} 特异度={tn/max(tn+fp,1):.2f}')

## 2. 真实数据检测

以 NASA B0005 为例（路径按本机修改）。

In [ ]:
from plating.readers import load
path = r'D:\项目\02_数据集\01_国际公开集\NASA_PCoE\1. BatteryAgingARC-FY08Q4\B0005.mat'
real = load('nasa', path)[0]
res = run_all(real, cfg)
for m in METHOD_NAMES:
    n = sum(1 for r in res[m] if r.positive)
    print(f'{m}: {n} 阳性循环')

## 3. 查看单条判定证据

In [ ]:
r = res['M3'][20]
print(r.method, r.cycle, r.positive, r.score, r.detail)